# 🚗 ADAS TSR: Weather-Adaptive Image Quality Monitor (IEEE Std 2020™-2024)

Notebook này hiện thực hóa bộ giám sát chất lượng ảnh trực tuyến (**In-line Image Quality Monitor**) đạt chuẩn quốc tế **IEEE Std 2020™-2024** chuyên biệt cho các hệ thống ADAS ô tô và xe máy.

### Mục tiêu nghiên cứu:
1. **Định lượng hóa chất lượng hình ảnh** dưới các điều kiện thời tiết phức tạp ở Việt Nam (mưa mờ, sương mù, chói sáng ban đêm).
2. **Kiểm soát rủi ro SOTIF (ISO 21448)**: Tự động kích hoạt các hành động phòng vệ (như chế độ hiển thị an toàn suy giảm *Unavailable* hoặc yêu cầu kiểm chứng chéo với bản đồ số) khi chất lượng ảnh camera sụt giảm dưới ngưỡng an toàn, ngăn chặn rủi ro người lái lạm dụng tính năng do chủ quan.
3. **Tương thích phần cứng biên**: Thuật toán được tối ưu hóa chạy cực mượt trên CPU nhúng (Jetson Nano, ECU) mà không cần cài đặt các thư viện nặng như Scipy.

## 🛠️ Step 1: Thiết lập môi trường và Import thư viện

In [ ]:
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt

print("Môi trường đã sẵn sàng! OpenCV version:", cv2.__version__)

## 🛠️ Step 2: Định nghĩa lớp Giám sát Chất lượng Ảnh IEEE2020CTAMonitorV2

Lớp này đo lường:
- **Contrast Transfer Accuracy (CTA)**: Sử dụng công thức Michelson Contrast cục bộ.
- **Contrast Signal-to-Noise Ratio (CSNR)**: Sử dụng phương pháp tách vùng **Foreground/Background Ring** và đo nhiễu độc lập ở các góc phẳng nhằm loại bỏ hiện tượng rò rỉ cạnh chữ số.
- **Contrast Detection Probability (CDP)**: Xác suất phân biệt tương phản thống kê (Normal CDF).

In [ ]:
class IEEE2020CTAMonitorV2:
    def __init__(self, cta_threshold=0.35, csnr_threshold=4.0, sharpness_threshold=50.0):
        self.cta_threshold = cta_threshold          # Ngưỡng tương phản Michelson tối thiểu
        self.csnr_threshold = csnr_threshold        # Ngưỡng CSNR tối thiểu
        self.sharpness_threshold = sharpness_threshold  # Ngưỡng sắc nét tránh mưa mờ
        
    def _get_luminance(self, bgr_img):
        """Chuyển đổi ảnh sang kênh Y (Luminance) sử dụng trọng số chuẩn ITU-R BT.601"""
        gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
        return gray.astype(np.float32)

    def _normal_cdf(self, x):
        """Hàm phân phối tích lũy chuẩn (Normal CDF) toán học thuần"""
        return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

    def analyze_roi(self, frame, bbox):
        """
        Phân tích vùng biển báo (ROI Bounding Box) để tính toán chất lượng ảnh cục bộ.
        bbox format: (x1, y1, x2, y2)
        """
        x1, y1, x2, y2 = map(int, bbox)
        h, w, _ = frame.shape
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        
        if (x2 - x1) < 10 or (y2 - y1) < 10:
            return {
                "status": "INVALID_ROI", 
                "cta": 0.0, 
                "csnr": 0.0, 
                "cdp": 0.0, 
                "sharpness": 0.0,
                "sotif_action": "IGNORE"
            }
            
        roi = frame[y1:y2, x1:x2]
        roi_y = self._get_luminance(roi)
        
        # 1. Tính Contrast Transfer Accuracy (CTA) qua công thức Michelson Contrast
        l_min, l_max, _, _ = cv2.minMaxLoc(roi_y)
        denom = (l_max + l_min)
        michelson_contrast = (l_max - l_min) / denom if denom > 0 else 0.0
        
        # 2. Tính Contrast Signal-to-Noise Ratio (CSNR) bằng phương pháp phân vùng Ring
        rh, rw = roi_y.shape
        cy, cx = rh // 2, rw // 2
        dy, dx = int(rh * 0.25), int(rw * 0.25)
        
        fg_mask = np.zeros_like(roi_y, dtype=np.uint8)
        cv2.rectangle(fg_mask, (cx - dx, cy - dy), (cx + dx, cy + dy), 255, -1)
        bg_mask = cv2.bitwise_not(fg_mask)
        
        fg_pixels = roi_y[fg_mask == 255]
        bg_pixels = roi_y[bg_mask == 255]
        
        if len(fg_pixels) > 0 and len(bg_pixels) > 0:
            mean_fg = np.mean(fg_pixels)
            mean_bg = np.mean(bg_pixels)
            
            # Ước lượng nhiễu cảm biến tại góc 5x5 phẳng để tránh nhiễu do cạnh sắc chữ số
            corner_block = roi_y[:5, :5]
            std_noise = np.std(corner_block)
            std_noise = max(std_noise, 0.5) # Tránh lỗi chia cho 0
            
            contrast_diff = abs(mean_fg - mean_bg)
            csnr = contrast_diff / std_noise
            
            # 3. Tính Contrast Detection Probability (CDP) - Ngưỡng phân biệt T = 5.0 độ sáng
            T = 5.0
            cdp = self._normal_cdf((contrast_diff - T) / std_noise)
        else:
            csnr, cdp = 0.0, 0.0
            
        # 4. Tính độ sắc nét (Sharpness Proxy) qua phương sai toán tử Laplacian
        sharpness = cv2.Laplacian(roi_y, cv2.CV_32F).var()
        
        # Logic ra quyết định SOTIF dựa trên các KPIs chất lượng ảnh
        is_cta_ok = michelson_contrast >= self.cta_threshold
        is_csnr_ok = csnr >= self.csnr_threshold
        is_sharp_ok = sharpness >= self.sharpness_threshold
        
        if is_cta_ok and is_csnr_ok and is_sharp_ok:
            status = "SAFE"
            sotif_action = "DISPLAY_CONFIRMED"
        elif not is_cta_ok or not is_csnr_ok:
            status = "UNSAFE_RAIN_OR_GLARE"
            sotif_action = "DEGRADED_VERIFICATION_REQUIRED"
        else:
            status = "DEGRADED_BLUR"
            sotif_action = "HOLD_PREVIOUS_DECISION"
            
        return {
            "status": status,
            "cta": float(michelson_contrast),
            "csnr": float(csnr),
            "cdp": float(cdp),
            "sharpness": float(sharpness),
            "sotif_action": sotif_action
        }

## 🌧️ Step 3: Bộ giả lập điều kiện thời tiết thực tế tại Việt Nam

In [ ]:
def create_synthetic_sign():
    """Tạo một khung ảnh biển báo giới hạn tốc độ 50 lý tưởng"""
    img = np.ones((300, 300, 3), dtype=np.uint8) * 128 # Nền đường xám trung tính
    cv2.circle(img, (150, 150), 100, (0, 0, 255), -1) # Vòng tròn đỏ
    cv2.circle(img, (150, 150), 80, (255, 255, 255), -1) # Nền trắng
    cv2.putText(img, "50", (100, 175), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 0, 0), 6, cv2.LINE_AA) # Số 50
    # Nhiễu cảm biến nhẹ
    noise = np.random.normal(0, 1.0, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

def simulate_heavy_rain_or_fog(img):
    """Mô phỏng sương mù dày đặc và mưa làm mờ nhòe ống kính (Blur + Giảm tương phản mạnh + Nhiễu hạt)"""
    blurred = cv2.GaussianBlur(img, (19, 19), 7)
    foggy = (blurred.astype(np.float32) * 0.25 + 95).astype(np.uint8)
    noise = np.random.normal(0, 5.0, foggy.shape).astype(np.int16)
    return np.clip(foggy.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def simulate_headlight_glare(img):
    """Mô phỏng ánh sáng chói lóa từ đèn pha xe đi ngược chiều chiếu trực diện ban đêm"""
    glare_img = img.copy()
    cv2.circle(glare_img, (120, 120), 120, (255, 255, 255), -1) # Vùng lóa trắng hình tròn
    noise = np.random.normal(0, 3.5, glare_img.shape).astype(np.int16)
    return np.clip(glare_img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

## 📊 Step 4: Chạy Thực nghiệm đo đạc và Trực quan hóa kết quả (Visualization)

In [ ]:
monitor = IEEE2020CTAMonitorV2()
bbox = (50, 50, 250, 250) # Vùng bounding box biển báo

img_clear = create_synthetic_sign()
img_rain = simulate_heavy_rain_or_fog(img_clear)
img_glare = simulate_headlight_glare(img_clear)

metrics_clear = monitor.analyze_roi(img_clear, bbox)
metrics_rain = monitor.analyze_roi(img_rain, bbox)
metrics_glare = monitor.analyze_roi(img_glare, bbox)

cases = [img_clear, img_rain, img_glare]
titles = ["Clear (Ly tuong)", "Heavy Rain/Fog (Mua mo)", "Headlight Glare (Loa sang)"]
metrics = [metrics_clear, metrics_rain, metrics_glare]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for i, ax in enumerate(axes):
    ax.imshow(cv2.cvtColor(cases[i], cv2.COLOR_BGR2RGB))
    ax.set_title(titles[i], fontsize=14, fontweight='bold')
    ax.axis("off")
    
    m = metrics[i]
    text_info = (
        f"Status: {m['status']}\n"
        f"SOTIF: {m['sotif_action']}\n"
        f"-----------------\n"
        f"CTA: {m['cta']:.3f}\n"
        f"CSNR: {m['csnr']:.2f}\n"
        f"CDP: {m['cdp']:.2%}\n"
        f"Sharpness: {m['sharpness']:.1f}"
    )
    text_color = "green" if m['status'] == "SAFE" else "red"
    ax.text(10, 280, text_info, color=text_color, fontsize=10, 
            bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()